# Base Cadastral


**Objetivo desta etapa:** entender a estrutura da base, validar sua qualidade e salvá-la limpa para a construção da ABT.

**Entrada:** `data/raw/base_cadastral.parquet`
**Saída:** `data/processed/base_cadastral.parquet`

**Resumo dos resultados**
- A base tinha 600 `id_cliente` duplicados (50.600 registros). Foi mantido um registro por cliente, resultando em **50.000 clientes únicos**.
- Textos foram padronizados (caixa, espaços e acentos) e a renda, em formato brasileiro, foi convertida para número.
- Valores impossíveis ou inválidos foram convertidos em ausente (NaN), **sem remover clientes**, e cada caso recebeu uma flag.
- `ocupacao` tinha 31% de ausentes, em parte estruturais (aposentados e desempregados).
- Imputação **não** foi feita nesta etapa, para evitar vazamento de dados (ver "Valores ausentes").

## Dicionário da Base Cadastral


| Feature                       | Descrição                                                          |
|-------------------------------|--------------------------------------------------------------------|
| `id_cliente`                  | Identificador único do cliente                                     |
| `sexo`                        | Sexo do cliente                                                    |
| `data_nascimento`             | Data de nascimento do cliente                                      |
| `qtd_filhos`                  | Quantidade de filhos declarados                                    |
| `qtd_membros_familia`         | Quantidade total de membros da família                             |
| `renda_anual`                 | Renda total anual declarada pelo cliente                           |
| `ocupacao`                    | Ocupação profissional do cliente                                   |
| `tipo_organizacao`            | Tipo de empresa que o cliente trabalha                             |
| `nivel_educacao`              | Nível de educação do cliente                                       |
| `estado_civil`                | Estado civil do cliente                                            |
| `tipo_moradia`                | Tipo de moradia                                                    |
| `possui_carro`                | Indicador se o cliente possui carro                                |
| `possui_imovel`               | Indicador se o cliente possui imóvel                               |
| `nota_regiao_cliente`         | Avaliação de risco da região do cliente                            |
| `nota_regiao_cliente_cidade`  | Avaliação de risco da região, ajustada para a cidade               |
| `tipo_renda`                  | Tipo de fonte de renda (trabalho, pensão, aposentadoria, etc.)     |


## Importação de Bibliotecas e Leitura dos Dados

In [2]:
import pandas as pd
import numpy as np
import pyarrow as pa
import re

In [3]:
df = pd.read_parquet('../../data/raw/base_cadastral.parquet')


## Dimensões


A base original tem **50.600 linhas e 16 colunas**. Depois da remoção dos duplicados, ficam **50.000 clientes**.

O dicionário da base descreve um registro único por cliente. Essa premissa não se sustenta e é verificada na seção seguinte.

In [4]:
df.shape

(50600, 16)

In [5]:
# Mostrar todas as colunas em ordem alfabética
print(sorted(df.columns))

['data_nascimento', 'estado_civil', 'id_cliente', 'nivel_educacao', 'nota_regiao_cliente', 'nota_regiao_cliente_cidade', 'ocupacao', 'possui_carro', 'possui_imovel', 'qtd_filhos', 'qtd_membros_familia', 'renda_anual', 'sexo', 'tipo_moradia', 'tipo_organizacao', 'tipo_renda']


In [6]:
df.head()

,id_cliente,sexo,data_nascimento,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade
0,100023,Feminino,1994-01-30,1,2.0,"90.000,00",Servidor público,Equipe administrativa,Educação infantil,Ensino superior,Solteiro,Casa ou apartamento,Não,Sim,2,2
1,100031,Feminino,1973-11-13,0,1.0,"112.500,00",Assalariado,Cozinheiros,Entidade empresarial tipo 3,Ensino médio,Viúvo,Casa ou apartamento,Não,Sim,3,2
2,100056,Masculino,1975-02-19,0,2.0,"360.000,00",Assalariado,Operários,Transporte tipo 2,Ensino médio,Casado,Casa ou apartamento,Sim,Sim,2,2
3,100069,Masculino,1986-04-10,1,2.0,"360.000,00",Assalariado,Operários,Transporte tipo 4,Ensino médio,Separado,Casa ou apartamento,Sim,Sim,2,2
4,100085,Masculino,1994-07-05,1,3.0,"-17.673,00",Assalariado,Motoristas,Entidade empresarial tipo 1,Ensino médio,Casado,Casa ou apartamento,Não,Sim,2,2


## Granularidade


Ao analisar a chave `id_cliente`, verificou-se que não há valores ausentes, porém 600 clientes aparecem duas vezes e nenhum aparece mais do que isso. 

Não há linhas completamente duplicadas: sempre há alguma diferença em pelo menos um campo, sendo mais comum na renda anual, estado civil e raramente em outros campos. Essa duplicidade pode gerar duplicação de contratos em joins futuros, então é importante considerar um tratamento adequado antes de usar esta base como referência principal.

Nos casos verificados, a renda do primeiro registro tinha valor "redondo" e a do segundo incluía centavos, sem padrão de atualização. Como a base não indica qual informação é a mais recente e não era viável revisar todos individualmente, optou-se por manter apenas o primeiro registro de cada cliente, removendo as duplicatas. Assim, evitamos eliminar clientes possivelmente relevantes ou criar médias artificiais. Há limitação de casos raros em que sexo ou data de nascimento também diferem, possivelmente por colisão de ID, mas não foram tratados pontualmente.

In [7]:
 
num_duplicated = df['id_cliente'].duplicated().sum()
print(f"Número de id_cliente duplicados: {num_duplicated}")

num_missing = df['id_cliente'].isnull().sum()
print(f"Número de id_cliente ausentes: {num_missing}")

Número de id_cliente duplicados: 600
Número de id_cliente ausentes: 0


In [8]:

duplicated_ids = df[df['id_cliente'].duplicated(keep=False)]

if not duplicated_ids.empty:
    id_cliente_exemplo = duplicated_ids['id_cliente'].iloc[0]

    linhas_exemplo = df[df['id_cliente'] == id_cliente_exemplo]
    
    print(f"Comparando registros para id_cliente {id_cliente_exemplo}:")
    display(linhas_exemplo)
else:
    print("Não há id_cliente duplicado para comparar.")

Comparando registros para id_cliente 100219:


,id_cliente,sexo,data_nascimento,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade
14,100219,Masculino,1994-07-03,1,3.0,"315.000,00",Associado comercial,Equipe administrativa,Entidade empresarial tipo 3,Ensino superior,Casado,Casa ou apartamento,Não,Sim,1,1
50152,100219,Masculino,1994-07-03,1,3.0,"364.681,49",Associado comercial,Equipe administrativa,Entidade empresarial tipo 3,Ensino superior,Não informado,Casa ou apartamento,Não,Sim,1,1


In [9]:

num_duplicated_ids = df['id_cliente'].duplicated().sum()
ids_duplicated_count = df['id_cliente'].value_counts()
ids_with_more_than2 = (ids_duplicated_count > 2).sum()
print(f"Número de id_cliente duplicados (aparecem pelo menos 2x): {num_duplicated_ids}")
print(f"Número de id_cliente que aparecem mais de 2 vezes: {ids_with_more_than2}")

duplicated_df = df[df['id_cliente'].duplicated(keep=False)]
cols_to_check = [col for col in df.columns if col != 'id_cliente']
variation_counts = {}

for col in cols_to_check:
    variation = duplicated_df.groupby('id_cliente')[col].nunique()
    n_with_variation = (variation > 1).sum()
    if n_with_variation > 0:
        variation_counts[col] = n_with_variation

if variation_counts:
    print("Colunas que variam dentro dos duplicados e a quantidade de IDs em que variam:")
    for col, count in variation_counts.items():
        print(f"- {col}: {count}")
else:
    print("Nenhuma coluna, além de id_cliente, varia dentro dos duplicados.")

Número de id_cliente duplicados (aparecem pelo menos 2x): 600
Número de id_cliente que aparecem mais de 2 vezes: 0
Colunas que variam dentro dos duplicados e a quantidade de IDs em que variam:
- sexo: 2
- data_nascimento: 6
- qtd_filhos: 3
- qtd_membros_familia: 4
- renda_anual: 600
- tipo_renda: 1
- estado_civil: 150
- possui_carro: 4
- possui_imovel: 10


In [10]:
df = df.drop_duplicates(subset=['id_cliente'], keep='first')

## Tipos e conversões

`data_nascimento` e `renda_anual` chegaram como texto (`object`). As conversões abaixo foram necessárias para uso numérico e temporal.


In [11]:
df.dtypes

id_cliente                      int64
sexo                           object
data_nascimento                object
qtd_filhos                      int64
qtd_membros_familia           float64
renda_anual                    object
tipo_renda                     object
ocupacao                       object
tipo_organizacao               object
nivel_educacao                 object
estado_civil                   object
tipo_moradia                   object
possui_carro                   object
possui_imovel                  object
nota_regiao_cliente             int64
nota_regiao_cliente_cidade      int64
dtype: object

### data_nascimento

 Quase todos os valores de `data_nascimento` já estavam no formato correto, com exceção de 37 casos usando a data inexistente `31/02/1990`, aparentemente como código para ausente. Converti a coluna para datetime usando `errors='coerce'`, transformando essas datas inválidas em `NaT` e marquei com uma flag (`data_nascimento_invalida`). Assim evito idades erradas ou erros em cálculos futuros.


In [12]:


date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")

data_nasc_strings = df['data_nascimento'].astype(str)
formatos = data_nasc_strings.apply(
    lambda x: 'YYYY-MM-DD' if date_pattern.match(x) else x
)

contagem_formatos = formatos.value_counts()

print("Quantidade de valores por formato encontrado em 'data_nascimento':")
for formato, quantidade in contagem_formatos.items():
    if formato == 'YYYY-MM-DD':
        print(f"- {formato} (esperado): {quantidade}")
    else:
        print(f"- {formato}: {quantidade}")

for formato in contagem_formatos.index:
    if formato != 'YYYY-MM-DD':
        exemplos = df[data_nasc_strings == formato]
        print(f"\nExemplos para formato diferente '{formato}':")
        print(exemplos[['data_nascimento']].head())
   

Quantidade de valores por formato encontrado em 'data_nascimento':
- YYYY-MM-DD (esperado): 49963
- 31/02/1990: 37

Exemplos para formato diferente '31/02/1990':
     data_nascimento
280       31/02/1990
1062      31/02/1990
1212      31/02/1990
2374      31/02/1990
2443      31/02/1990


In [13]:
df['data_nascimento'] = pd.to_datetime(df['data_nascimento'], format='%Y-%m-%d', errors='coerce')  # 31/02 vira NaT
df['data_nascimento_invalida'] = df['data_nascimento'].isna().astype(int)



### qtd_membros_familia

Vem como `float64`, mas não tem valores fracionários nem ausentes. A mudança de tipo não altera nenhum valor.

In [14]:

print("Valores únicos em 'qtd_membros_familia':")
print(df['qtd_membros_familia'].unique())

nao_inteiros = df[
    df['qtd_membros_familia'].notnull() & 
    (df['qtd_membros_familia'] % 1 != 0)
]
if not nao_inteiros.empty:
    print("\nValores fracionários encontrados em 'qtd_membros_familia':")
    print(nao_inteiros[['qtd_membros_familia']].drop_duplicates())
else:
    print("\nNenhum valor fracionário encontrado em 'qtd_membros_familia'.")

nulos = df[df['qtd_membros_familia'].isnull()]
if not nulos.empty:
    print(f"\nQuantidade de valores ausentes em 'qtd_membros_familia': {nulos.shape[0]}")

Valores únicos em 'qtd_membros_familia':
[ 2.  1.  3.  4.  5.  0.  6. 10.  9.  8. 14.  7. 15.]

Nenhum valor fracionário encontrado em 'qtd_membros_familia'.


In [15]:
df['qtd_membros_familia'] = df['qtd_membros_familia'].fillna(0).astype(int)


### renda_anual

Está em formato brasileiro (`315.000,00`). Foram removidos o separador de milhar e o símbolo `R$`, a vírgula virou ponto decimal e o texto `não informado` virou NaN (**98 clientes**).


In [16]:
df['renda_anual'] = (
    df['renda_anual']
    .astype(str)
    .replace('não informado', np.nan)
    .str.replace('R$', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

## Variáveis Categóricas

Observou-se que várias categorias estavam escritas de formas diferentes, com variações de caixa, acentuação e espaços extras, principalmente nas variáveis `sexo`, `tipo_renda`, `possui_carro` e `possui_imovel`. Para corrigir essas inconsistências, os valores foram padronizados, usando transformação para minúsculas e remoção de espaços, e depois convertidos para rótulos únicos e padronizados. Valores fora dos esperados nessas variáveis foram classificados como "Não informado".


In [17]:
# Mostrando todas as variáveis categóricas do DataFrame
variaveis_categoricas = df.select_dtypes(include=['object']).columns.tolist()
print("Variáveis categóricas:")
print(variaveis_categoricas)

Variáveis categóricas:
['sexo', 'tipo_renda', 'ocupacao', 'tipo_organizacao', 'nivel_educacao', 'estado_civil', 'tipo_moradia', 'possui_carro', 'possui_imovel']


### Sexo

In [18]:
print("Valores únicos de 'sexo':")
print(df['sexo'].unique())

Valores únicos de 'sexo':
['Feminino' 'Masculino' 'MASCULINO' 'Feminino ' 'Não informado' 'feminino']


In [19]:
df['sexo'] = (
    df['sexo']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'masculino': 'Masculino',
        'feminino': 'Feminino',
        'não informado': 'Não informado'
    })
)

df.loc[~df['sexo'].isin(['Masculino', 'Feminino', 'Não informado']), 'sexo'] = 'Não informado'

### Tipo Renda

In [20]:

print("Valores únicos de 'tipo_renda':")
print(df['tipo_renda'].unique())

Valores únicos de 'tipo_renda':
['Servidor público' 'Assalariado' 'Associado comercial' 'Aposentado'
 'Desempregado' 'ASSALARIADO' 'Não informado' 'assalariado' 'Autonomo'
 'Estudante' 'Empresário' 'Licença maternidade' 'Autônomo']


In [21]:
df['tipo_renda'] = df['tipo_renda'].replace(
    ['ASSALARIADO', 'assalariado', 'Assalariado'], 'Assalariado'
)
df['tipo_renda'] = df['tipo_renda'].replace(
    ['AUTONOMO', 'Autonomo', 'autonomo', 'Autônomo', 'autônomo'], 'Autônomo'
)

### Possui Carro

In [21]:

print(df['possui_carro'].unique())

['Não' 'Sim' 'sim' 'SIM' 'nao' 'não informado' 'Não ']


In [22]:
df['possui_carro'] = (
    df['possui_carro']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'sim': 'Sim',
        'não': 'Não',
        'nao': 'Não',
        'não informado': 'Não informado'
    })
)

df.loc[~df['possui_carro'].isin(['Sim', 'Não', 'Não informado']), 'possui_carro'] = 'Não informado'

### Possui Imovel

In [23]:

print(df['possui_imovel'].unique())

['Sim' 'Não' 'sim' 'Não ' 'não informado' 'SIM' 'nao']


In [24]:
df['possui_imovel'] = (
    df['possui_imovel']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'sim': 'Sim',
        'não': 'Não',
        'nao': 'Não',
        'não informado': 'Não informado'
    })
)

df.loc[~df['possui_imovel'].isin(['Sim', 'Não', 'Não informado']), 'possui_imovel'] = 'Não informado'

## Variavéis Númericas

As variáveis numéricas apresentaram alguns pontos relevantes: grande parte dos clientes não tem filhos (média 0,49, 75% até 1) e famílias costumam ter 2 pessoas, mas houve casos incongruentes com zero membros. 

A renda anual apresentou valores negativos e extremos, distorcendo a média e o desvio padrão,  os negativos e zeros foram marcados como NaN para indicar invalidação, enquanto valores muito altos foram mantidos, já que não há evidência. As variáveis de nota de região são categorias ordinais de 1 a 3, altamente concentradas no valor 2, e não foram alteradas. As demais variáveis serão discutidas nas seções seguintes.

In [25]:
numeric_cols = df.select_dtypes(include=['number']).columns.drop('id_cliente')
df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
qtd_filhos,50000.0,0.490940,0.837527,0.0,0.0,0.0,1.0,14.0
qtd_membros_familia,50000.0,2.255800,0.992545,0.0,2.0,2.0,3.0,15.0
renda_anual,49902.0,199986.265243,978901.993546,-49832.0,112500.0,148500.0,202500.0,117000000.0
nota_regiao_cliente,50000.0,2.084000,0.539657,1.0,2.0,2.0,2.0,3.0
nota_regiao_cliente_cidade,50000.0,2.072280,0.533724,1.0,2.0,2.0,2.0,3.0
data_nascimento_invalida,50000.0,0.000740,0.027193,0.0,0.0,0.0,0.0,1.0


### Renda Anual

Na análise da renda anual, foram encontradas entradas negativas, zeradas e "não informado", além de valores muito altos acima de 1 milhão. Optou-se por marcar como NaN os negativos e zeros, sinalizando esses casos com flags adicionais para futura referência, pois não faz sentido substituí-los por outro valor. 

Os casos de "não informado" também viraram NaN, mas com flag específica de ausência. Já os casos extremos foram mantidos, pois não havia evidência suficiente de erro. 

No total, cerca de 0,5% dos dados ficaram como NaN devido a essas situações.

In [26]:
print("negativas:", (df['renda_anual'] < 0).sum(), "| zeros:", (df['renda_anual'] == 0).sum())


negativas: 66 | zeros: 67


In [27]:
print(df['renda_anual'].quantile([.01, .05, .95, .99, .999]))


0.010    4.500000e+04
0.050    6.750000e+04
0.950    3.495000e+05
0.990    5.400000e+05
0.999    1.502596e+07
Name: renda_anual, dtype: float64


In [28]:
cols = ['id_cliente', 'renda_anual', 'tipo_renda', 'ocupacao', 'tipo_organizacao']
df.nlargest(10, 'renda_anual')[cols]

,id_cliente,renda_anual,tipo_renda,ocupacao,tipo_organizacao
1397,114967,117000000.0,Assalariado,Operários,Entidade empresarial tipo 3
18085,286003,34027248.0,Aposentado,None,Não informado
49386,465635,33411430.0,Assalariado,None,Outro
3114,132401,33350279.0,Servidor público,Equipe administrativa,Ministérios de segurança
29046,399547,33123947.0,Assalariado,Operários,Entidade empresarial tipo 1
25213,359923,32856942.0,Aposentado,None,Não informado
29468,403788,32454677.0,Associado comercial,Contadores,Entidade empresarial tipo 3
12990,234497,31937246.0,Assalariado,Equipe administrativa,Educação infantil
14933,253700,31864156.0,Assalariado,None,Comércio tipo 2
45810,462059,31678511.0,Associado comercial,Gerentes,Entidade empresarial tipo 3


In [29]:
df[df['renda_anual'] < 0][cols]

,id_cliente,renda_anual,tipo_renda,ocupacao,tipo_organizacao
4,100085,-17673.0,Assalariado,Motoristas,Entidade empresarial tipo 1
305,103590,-40225.0,Aposentado,None,Não informado
2380,125198,-23842.0,Associado comercial,Equipe administrativa,Comércio tipo 3
3170,132986,-45254.0,Associado comercial,None,Construção
4881,150413,-49832.0,Assalariado,Gerentes,Entidade empresarial tipo 3
...,...,...,...,...,...
47680,463929,-38796.0,Assalariado,Operários,Governo
47971,464220,-37279.0,Autônomo,Operários,Entidade empresarial tipo 3
48926,465175,-30587.0,Assalariado,None,Não informado
49078,465327,-36900.0,Aposentado,Profissionais de saúde,Comércio tipo 7


In [30]:
neg = df[df['renda_anual'] < 0]
print(pd.concat([neg['tipo_renda'].value_counts(normalize=True).rename('negativos'),
                 df['tipo_renda'].value_counts(normalize=True).rename('base')], axis=1).round(3))
print(30*"-")
print(neg['renda_anual'].describe())
print(30*"-")
print(df[df['renda_anual'] == 0]['tipo_renda'].value_counts())

                     negativos   base
tipo_renda                           
Assalariado              0.530  0.506
Associado comercial      0.227  0.229
Aposentado               0.182  0.183
Servidor público         0.045  0.073
Autônomo                 0.015  0.008
Não informado              NaN  0.001
Desempregado               NaN  0.000
Licença maternidade        NaN  0.000
Estudante                  NaN  0.000
Empresário                 NaN  0.000
------------------------------
count       66.000000
mean    -28844.393939
std      12311.187492
min     -49832.000000
25%     -38600.750000
50%     -29832.000000
75%     -18693.250000
max      -2133.000000
Name: renda_anual, dtype: float64
------------------------------
tipo_renda
Assalariado            35
Associado comercial    16
Aposentado             11
Servidor público        4
Autônomo                1
Name: count, dtype: int64


In [31]:
print(df['renda_anual'].quantile([.99, .995, .998, .999, .9995, 1]))
print(30*"-")
alta = df[df['renda_anual'] > 1_000_000]
print(len(alta))
print(30*"-")
print(alta['tipo_renda'].value_counts(normalize=True).round(3))
print(30*"-")
print(alta['ocupacao'].value_counts().head(8))


0.9900    5.400000e+05
0.9950    6.750000e+05
0.9980    9.989100e+05
0.9990    1.502596e+07
0.9995    2.424120e+07
1.0000    1.170000e+08
Name: renda_anual, dtype: float64
------------------------------
100
------------------------------
tipo_renda
Assalariado            0.42
Associado comercial    0.33
Aposentado             0.17
Servidor público       0.07
Empresário             0.01
Name: proportion, dtype: float64
------------------------------
ocupacao
Gerentes                   22
Operários                  15
Motoristas                  7
Equipe administrativa       7
Técnicos especializados     6
Equipe de vendas            6
Contadores                  4
Profissionais de saúde      4
Name: count, dtype: int64


### Familia e Filhos

Na análise de `qtd_membros_familia`, encontramos registros com valor 0, o que não faz sentido (mínimo esperado é 1, considerando o cliente). Esses casos foram marcados com uma flag (`membros_invalido`) e convertidos para NaN, já que provavelmente são inválidos.

Para `qtd_filhos`, a maioria tem zero filhos e valores muito altos são raros, mas plausíveis. Não foram feitas alterações nesses casos. Casos em que há mais filhos que membros de família também foram mantidos, pois são possíveis (filhos que moram em outros lugares).

In [32]:
print("membros == 0:", (df['qtd_membros_familia'] == 0).sum())

membros == 0: 98


In [33]:

print(df['qtd_filhos'].value_counts().sort_index())

qtd_filhos
0     33685
1     10117
2      4799
3       983
4       279
5        89
6        34
7         5
8         6
9         1
12        1
14        1
Name: count, dtype: int64


In [34]:

inconsistente = df['qtd_membros_familia'] < (df['qtd_filhos'])
print("membros < filhos + 1:", inconsistente.sum())
print(df[inconsistente][['id_cliente', 'qtd_filhos', 'qtd_membros_familia', 'estado_civil']].head(10))


membros < filhos + 1: 135
      id_cliente  qtd_filhos  qtd_membros_familia estado_civil
534       105950           6                    2       Casado
599       106600           1                    0     Solteiro
939       110051           1                    0       Casado
1239      113345           6                    3       Casado
1800      119078           5                    1     Solteiro
1934      120318           5                    1        Viúvo
2771      128983           5                    2       Casado
3080      132044           1                    0       Casado
3178      133099           1                    0       Casado
3714      138440           1                    0       Casado


In [35]:
df['membros_invalido'] = (df['qtd_membros_familia'] == 0).astype(int)

In [36]:
df.loc[df['qtd_membros_familia'] == 0, 'qtd_membros_familia'] = np.nan

## Valores Ausentes

Algumas colunas apresentam valores ausentes ou inválidos. A coluna `ocupacao` tem cerca de 31% de nulos, que podem ser tanto por ausência estrutural (ex: aposentados e desempregados) quanto por falta de preenchimento, detalhes em outra seção. 

A `renda_anual` tem poucos ausentes (0,46%), relacionados tanto a registros explicitamente como "não informado" quanto valores negativos ou zero; nesses casos, os NaNs foram mantidos e flags criadas quando relevante. 

Para `qtd_membros_familia`, registros com valor zero (0,20%) foram considerados inválidos e tratados como NaN, também com flag. Já `data_nascimento` apresenta poucos casos de datas inválidas (como 31/02/1990), que foram convertidos para `NaT` e sinalizados. As demais colunas não possuem nulos, mas várias usam "Não informado" como uma das categorias.


In [37]:
percent_missing = df.isnull().mean() * 100
print("Porcentagem de valores ausentes por coluna:")
print(percent_missing.sort_values(ascending=False))

Porcentagem de valores ausentes por coluna:
ocupacao                      31.022
renda_anual                    0.196
qtd_membros_familia            0.196
data_nascimento                0.074
sexo                           0.000
id_cliente                     0.000
qtd_filhos                     0.000
tipo_renda                     0.000
tipo_organizacao               0.000
nivel_educacao                 0.000
estado_civil                   0.000
tipo_moradia                   0.000
possui_carro                   0.000
possui_imovel                  0.000
nota_regiao_cliente            0.000
nota_regiao_cliente_cidade     0.000
data_nascimento_invalida       0.000
membros_invalido               0.000
dtype: float64


### Ocupação

No processamento da coluna `ocupacao`, identificamos que valores ausentes podem ocorrer por razões diferentes: no caso de aposentados e desempregados, a ausência é estrutural (não possuem ocupação ativa), então imputamos como "Aposentado" e "Sem Ocupação", respectivamente. 

Para os demais casos em que a ocupação deveria existir mas não foi preenchida (ex: assalariado, associado comercial, servidor público, autônomo), preenchido como "Não informado". Assim, asseguramos que cada situação tenha o tratamento mais adequado.



In [38]:
print("Valores únicos de 'ocupacao':")
print(df['ocupacao'].unique())

Valores únicos de 'ocupacao':
['Equipe administrativa' 'Cozinheiros' 'Operários' 'Motoristas'
 'Garçons e atendentes de bar' 'Equipe de vendas' 'Equipe de segurança'
 None 'Serviços particulares' 'Gerentes' 'Técnicos especializados'
 'Contadores' 'Operários de baixa qualificação' 'Profissionais de saúde'
 'Equipe de limpeza' 'Secretárias' 'Equipe de tecnologia'
 'Equipe de recursos humanos' 'Corretores imobiliários']


In [39]:
print("3 exemplos onde 'ocupacao' é ausente:")
df[df['ocupacao'].isnull()].head(10)

3 exemplos onde 'ocupacao' é ausente:


,id_cliente,sexo,data_nascimento,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade,data_nascimento_invalida,membros_invalido
9,100165,Feminino,1964-12-20,0,2.0,175500.0,Aposentado,None,Não informado,Ensino médio,Casado,Casa ou apartamento,Sim,Sim,2,2,0,0
10,100178,Feminino,1972-04-23,0,1.0,119250.0,Assalariado,None,Autônomo,Ensino superior,Separado,Casa ou apartamento,Não,Sim,2,2,0,0
11,100193,Feminino,1976-03-06,0,1.0,225000.0,Associado comercial,None,Outro,Ensino médio,Separado,Casa ou apartamento,Sim,Não,3,3,0,0
12,100200,Feminino,1978-11-13,0,1.0,216000.0,Assalariado,None,Entidade empresarial tipo 3,Ensino médio,Viúvo,Casa ou apartamento,Não,Não,2,2,0,0
15,100233,Feminino,1974-11-19,0,2.0,216000.0,Assalariado,None,Transporte tipo 4,Ensino médio,Casado,Casa ou apartamento,Não,Sim,2,2,0,0
19,100266,Feminino,1973-08-23,0,1.0,157500.0,Associado comercial,None,Entidade empresarial tipo 3,Ensino médio,Viúvo,Apartamento municipal,Não,Não,1,1,0,0
27,100332,Feminino,1960-12-07,0,1.0,157500.0,Aposentado,None,Não informado,Ensino médio,Viúvo,Casa ou apartamento,Não,Sim,2,2,0,0
33,100406,Feminino,1968-01-12,0,2.0,135000.0,Aposentado,None,Não informado,Ensino médio,Casado,Casa ou apartamento,Não,Sim,2,2,0,0
37,100437,Feminino,1966-07-05,1,3.0,67500.0,Aposentado,None,Não informado,Ensino médio,Casado,Casa ou apartamento,Não,Sim,2,2,0,0
39,100458,Feminino,1968-07-29,0,2.0,90000.0,Aposentado,None,Não informado,Ensino médio,Casado,Casa ou apartamento,Não,Não,3,3,0,0


In [40]:
unique_tipo_renda = df.loc[df['ocupacao'].isnull(), 'tipo_renda'].unique()
print("Valores únicos de 'tipo_renda' quando 'ocupacao' é nulo:")
print(unique_tipo_renda)

Valores únicos de 'tipo_renda' quando 'ocupacao' é nulo:
['Aposentado' 'Assalariado' 'Associado comercial' 'Servidor público'
 'Desempregado' 'Não informado' 'Autônomo']


In [41]:
df.loc[df['ocupacao'].isnull() & (df['tipo_renda'] == 'Aposentado'), 'ocupacao'] = 'Aposentado'
df.loc[df['ocupacao'].isnull() & (df['tipo_renda'] == 'Desempregado'), 'ocupacao'] = 'Sem Ocupação'
df['ocupacao'] = df['ocupacao'].fillna('Não informado')

## Salvando Df Limpo

In [42]:

df.to_parquet('../../data/processed/base_cadastral.parquet', index=False)